In [2]:
import pandas as pd

df = pd.read_csv('data_sample_3.csv')
df.head()

,date,EstimatedCost,RawMaterial,Workmanship,StorageCost
0,1/1/19,1094,211,386,40
1,2/1/19,2850,523,199,49
2,3/1/19,2168,580,236,39
3,4/1/19,1370,368,559,34
4,5/1/19,2429,550,291,33


In [3]:
# Display basic info about the dataframe
print("Dataset shape:", df.shape)
print("\nData types:")
print(df.dtypes)
print("\nBasic statistics:")
print(df.describe())
print("\nMissing values:")
print(df.isnull().sum())
print("\nFirst few rows:")
print(df.head())

Dataset shape: (21, 5)

Data types:
date               str
EstimatedCost    int64
RawMaterial      int64
Workmanship      int64
StorageCost      int64
dtype: object

Basic statistics:
       EstimatedCost  RawMaterial  Workmanship  StorageCost
count      21.000000    21.000000    21.000000    21.000000
mean     1501.190476   410.190476   277.761905    39.761905
std       728.790136   192.720684   192.809985     7.535946
min       383.000000    21.000000    17.000000    29.000000
25%       930.000000   271.000000    60.000000    33.000000
50%      1486.000000   452.000000   276.000000    40.000000
75%      2168.000000   550.000000   441.000000    43.000000
max      2850.000000   673.000000   559.000000    55.000000

Missing values:
date             0
EstimatedCost    0
RawMaterial      0
Workmanship      0
StorageCost      0
dtype: int64

First few rows:
     date  EstimatedCost  RawMaterial  Workmanship  StorageCost
0  1/1/19           1094          211          386           40
1  2/1

In [4]:
# Calculate ActualCost, SoldPrice, and MarginOfProfit
df['ActualCost'] = df['RawMaterial'] + df['Workmanship'] + df['StorageCost']
df['SoldPrice'] = df['EstimatedCost'] * 1.1
df['MarginOfProfit'] = df['SoldPrice'] - df['ActualCost']

# Display the updated dataframe
df.head(10)

,date,EstimatedCost,RawMaterial,Workmanship,StorageCost,ActualCost,SoldPrice,MarginOfProfit
0,1/1/19,1094,211,386,40,637,1203.4,566.4
1,2/1/19,2850,523,199,49,771,3135.0,2364.0
2,3/1/19,2168,580,236,39,855,2384.8,1529.8
3,4/1/19,1370,368,559,34,961,1507.0,546.0
4,5/1/19,2429,550,291,33,874,2671.9,1797.9
5,6/1/19,1644,143,395,31,569,1808.4,1239.4
6,7/1/19,1641,640,497,30,1167,1805.1,638.1
7,8/1/19,2340,673,472,29,1174,2574.0,1400.0
8,9/1/19,2327,90,31,32,153,2559.7,2406.7
9,10/1/19,1486,398,528,33,959,1634.6,675.6


In [21]:
df.MarginOfProfit.describe()

count      21.000000
mean      923.595238
std       800.017577
min      -398.900000
25%       471.000000
50%       832.000000
75%      1400.000000
max      2406.700000
Name: MarginOfProfit, dtype: float64

In [31]:
import json
import pandas as pd
from IPython.display import HTML, display

# Select series for the multiline chart
series = ['EstimatedCost', 'ActualCost', 'SoldPrice', 'MarginOfProfit']
series = [s for s in series if s in df.columns]

# Prepare data (convert date to ISO string for D3 parsing)
df_plot = df.copy()
df_plot['date'] = pd.to_datetime(df_plot['date'], format='%m/%d/%y', errors='coerce')
df_plot = df_plot.dropna(subset=['date'])
chart_df = df_plot[['date'] + series].copy()
chart_df['date'] = chart_df['date'].dt.strftime('%Y-%m-%d')
records = chart_df.to_dict(orient='records')

chart_id = 'd3-multiline-chart'
table_id = 'd3-data-table'

html_template = """
<div class="dash">
  <div class="chart-wrap">
    <div id="__CHART_ID__"></div>
  </div>
  <div class="table-wrap">
    <div class="table-title">Data</div>
    <div class="table-scroll">
      <table id="__TABLE_ID__"></table>
    </div>
  </div>
</div>

<style>
  .dash {
    display: flex;
    gap: 18px;
    align-items: flex-start;
  }

  .chart-wrap { flex: 2; min-width: 680px; }
  .table-wrap {
    flex: 1;
    min-width: 360px;
    border: 1px solid #e2e8f0;
    border-radius: 10px;
    overflow: hidden;
    background: #fff;
  }
  .table-title {
    padding: 10px 12px;
    font: 600 14px/1.2 system-ui, -apple-system, Segoe UI, Roboto, sans-serif;
    border-bottom: 1px solid #e2e8f0;
    background: #f8fafc;
    color: #0f172a;
  }
  .table-scroll { max-height: 780px; overflow: auto; }

  table { width: 100%; border-collapse: collapse; font: 12px/1.3 system-ui, -apple-system, Segoe UI, Roboto, sans-serif; }
  th, td { padding: 6px 8px; border-bottom: 1px solid #eef2f7; text-align: right; white-space: nowrap; }
  th:first-child, td:first-child { text-align: left; }
  thead th {
    position: sticky; top: 0;
    background: #f1f5f9; color: #0f172a;
    z-index: 1; border-bottom: 1px solid #e2e8f0;
  }
  tbody tr:hover { background: #f8fafc; }

  .chart-title { font: 600 16px/1.2 system-ui, -apple-system, Segoe UI, Roboto, sans-serif; fill: #0b1f3a; }
  .axis path, .axis line { stroke: #94a3b8; }
  .grid line { stroke: #e2e8f0; }
  .legend text { font: 12px/1.2 system-ui, -apple-system, Segoe UI, Roboto, sans-serif; fill: #0f172a; }

  .tooltip {
    position: absolute;
    pointer-events: none;
    background: rgba(15, 23, 42, 0.92);
    color: white;
    padding: 10px 12px;
    border-radius: 10px;
    font: 12px/1.35 system-ui, -apple-system, Segoe UI, Roboto, sans-serif;
    box-shadow: 0 10px 25px rgba(0,0,0,0.18);
    min-width: 200px;
  }
  .tooltip .date { font-weight: 700; margin-bottom: 6px; }
  .tooltip .row { display: flex; justify-content: space-between; gap: 12px; }
  .tooltip .label { opacity: 0.9; }
  .tooltip .value { font-variant-numeric: tabular-nums; }
</style>

<script src="https://d3js.org/d3.v7.min.js"></script>
<script>
(() => {
  const data = __DATA__;
  const series = __SERIES__;

  // ---------------------------
  // Build Table (right side)
  // ---------------------------
  const table = d3.select('#__TABLE_ID__');
  table.selectAll('*').remove();
  const columns = ['date'].concat(series);

  table.append('thead').append('tr')
    .selectAll('th')
    .data(columns)
    .enter().append('th')
    .text(d => d);

  const tbody = table.append('tbody');
  const fmt = d3.format(',.2f');

  data.forEach(row => {
    const tr = tbody.append('tr');
    columns.forEach(col => {
      let v = row[col];
      if (col !== 'date') {
        const num = +v;
        v = Number.isFinite(num) ? fmt(num) : '';
      }
      tr.append('td').text(v);
    });
  });

  // ---------------------------
  // Build Chart (left side)
  // ---------------------------
  const container = d3.select('#__CHART_ID__');
  container.selectAll('*').remove();

  const margin = { top: 40, right: 120, bottom: 40, left: 60 };
  const width = 1000 - margin.left - margin.right;
  const height = 800 - margin.top - margin.bottom;

  const wrapper = container.append('div')
    .style('position', 'relative');

const svg = wrapper.append('svg')
  .attr('viewBox', '0 0 1000 800')
  .attr('preserveAspectRatio', 'xMidYMid meet')
  .style('width', '100%')
  .style('height', 'auto');

  const g = svg.append('g')
    .attr('transform', 'translate(' + margin.left + ',' + margin.top + ')');

  const parseDate = d3.timeParse('%Y-%m-%d');
  data.forEach(d => {
    d.date = parseDate(d.date);
    series.forEach(s => d[s] = +d[s]);
  });

  // Sort by date (important for bisector)
  data.sort((a, b) => a.date - b.date);

  const x = d3.scaleTime()
    .domain(d3.extent(data, d => d.date))
    .range([0, width]);

  const yMax = d3.max(series, s => d3.max(data, d => Number.isFinite(d[s]) ? d[s] : undefined)) ?? 0;

  const y = d3.scaleLinear()
    .domain([-500, yMax])
    .nice()
    .range([height, 0]);

  const color = d3.scaleOrdinal()
    .domain(series)
    .range(d3.schemeCategory10);

  const line = d3.line()
    .defined(d => Number.isFinite(d.value))
    .x(d => x(d.date))
    .y(d => y(d.value));

  const seriesData = series.map(name => ({
    name: name,
    values: data.map(d => ({ date: d.date, value: d[name] }))
  }));

  g.append('g')
    .attr('class', 'grid')
    .call(d3.axisLeft(y).ticks(6).tickSize(-width).tickFormat(''));

  g.append('g')
    .attr('class', 'axis')
    .attr('transform', 'translate(0,' + height + ')')
    .call(d3.axisBottom(x));

  g.append('g')
    .attr('class', 'axis')
    .call(d3.axisLeft(y));

  g.append('text')
    .attr('class', 'chart-title')
    .attr('x', 0)
    .attr('y', -12)
    .text('Project Financials Over Time');

  g.selectAll('.series')
    .data(seriesData)
    .join('path')
    .attr('class', 'series')
    .attr('fill', 'none')
    .attr('stroke', d => color(d.name))
    .attr('stroke-width', 2)
    .attr('d', d => line(d.values));

  const legend = g.append('g')
    .attr('class', 'legend')
    .attr('transform', 'translate(' + (width + 20) + ',0)');

  const legendItem = legend.selectAll('g')
    .data(series)
    .join('g')
    .attr('transform', (d, i) => 'translate(0,' + (i * 18) + ')');

  legendItem.append('rect')
    .attr('width', 12)
    .attr('height', 12)
    .attr('fill', d => color(d));

  legendItem.append('text')
    .attr('x', 18)
    .attr('y', 10)
    .text(d => d);

  // ---------------------------
  // Tooltip + Hover Interaction
  // ---------------------------
  const tooltip = wrapper.append('div')
    .attr('class', 'tooltip')
    .style('display', 'none');

  const focus = g.append('g')
    .style('display', 'none');

  // vertical line
  focus.append('line')
    .attr('class', 'focus-line')
    .attr('y1', 0)
    .attr('y2', height)
    .attr('stroke', '#334155')
    .attr('stroke-width', 1)
    .attr('stroke-dasharray', '3,3');

  // one circle per series
  const focusDots = focus.selectAll('circle')
    .data(series)
    .enter()
    .append('circle')
    .attr('r', 4)
    .attr('stroke', '#fff')
    .attr('stroke-width', 1.5)
    .attr('fill', d => color(d));

  const bisectDate = d3.bisector(d => d.date).left;
  const dateFmt = d3.timeFormat('%b %d, %Y');

  // overlay for capturing mouse
  g.append('rect')
    .attr('class', 'overlay')
    .attr('width', width)
    .attr('height', height)
    .attr('fill', 'transparent')
    .on('mouseenter', () => {
      focus.style('display', null);
      tooltip.style('display', null);
    })
    .on('mouseleave', () => {
      focus.style('display', 'none');
      tooltip.style('display', 'none');
    })
    .on('mousemove', (event) => {
      const [mx] = d3.pointer(event);
      const xDate = x.invert(mx);

      const i = bisectDate(data, xDate, 1);
      const d0 = data[i - 1];
      const d1 = data[i] || d0;
      const d = (xDate - d0.date) > (d1.date - xDate) ? d1 : d0;

      const fx = x(d.date);

      // move focus line
      focus.select('line')
        .attr('x1', fx)
        .attr('x2', fx);

      // move dots
      focusDots
        .attr('cx', fx)
        .attr('cy', s => y(d[s]));

      // tooltip content
      const rows = series.map(s => {
        const val = Number.isFinite(d[s]) ? fmt(d[s]) : '—';
        return '<div class="row"><span class="label">' + s + '</span><span class="value">' + val + '</span></div>';
      }).join('');

      tooltip.html(
        '<div class="date">' + dateFmt(d.date) + '</div>' + rows
      );

      // tooltip position (relative to wrapper)
      const pad = 14;
      const bbox = wrapper.node().getBoundingClientRect();
      const pageX = event.clientX - bbox.left;
      const pageY = event.clientY - bbox.top;

      // nudge right unless near edge
      const tW = 240;
      const left = (pageX + pad + tW > (margin.left + width + margin.right))
        ? (pageX - tW - pad)
        : (pageX + pad);

      const top = Math.max(0, pageY - 20);

      tooltip
        .style('left', left + 'px')
        .style('top', top + 'px');
    });

})();
</script>
"""

html = (html_template
        .replace('__DATA__', json.dumps(records))
        .replace('__SERIES__', json.dumps(series))
        .replace('__CHART_ID__', chart_id)
        .replace('__TABLE_ID__', table_id))

display(HTML(html))
print("HTML rendered")

HTML rendered


In [32]:
from pathlib import Path
out = Path("d3_Assignment_2.html")
out.write_text(html)

10931